# Generation evidence

This notebook checks the text-generation claims against the saved test predictions. It uses every held-out turn rather than a small demonstration set. The raw JSONL files are compressed under `results/predictions/`.

The evidence is mixed: the full fine-tune reliably produces Nepali news-shaped text, but fluent wording does not guarantee correct entities. A small number of greedy outputs also collapse into repetition.

In [1]:
import gzip
import json
import re
import statistics
import unicodedata
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

def read_gzip_jsonl(name):
    path = ROOT / 'results' / 'predictions' / name
    with gzip.open(path, 'rt', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle]

full = read_gzip_jsonl('full_predictions.jsonl.gz')
sampled = read_gzip_jsonl('sampled_predictions.jsonl.gz')
base_bundle = read_gzip_jsonl('base_predictions.jsonl.gz')
print(f'full: {len(full)} rows')
print(f'sampled: {len(sampled)} rows')
print(f'base bundle: {len(base_bundle)} rows (3 systems × 1718)')

full: 1718 rows
sampled: 1718 rows
base bundle: 5154 rows (3 systems × 1718)


In [2]:
def tokens(text):
    result, current = [], []
    for character in text.lower():
        if unicodedata.category(character)[0] in 'LMN':
            current.append(character)
        elif current:
            result.append(''.join(current))
            current = []
    if current:
        result.append(''.join(current))
    return result

def rep4(text):
    words = tokens(text)
    grams = [tuple(words[i:i + 4]) for i in range(len(words) - 3)]
    return 1 - len(set(grams)) / len(grams) if grams else 0.0

def summarize(rows):
    lengths = sorted(row['generated_tokens'] for row in rows)
    return {
        'median': int(statistics.median(lengths)),
        'p90': lengths[int(0.9 * (len(lengths) - 1))],
        'capped': sum(row['hit_max_new_tokens'] for row in rows),
        'empty': sum(not row['prediction'].strip() for row in rows),
        'repetitive': sum(rep4(row['prediction']) > 0.1 for row in rows),
        'ascii': sum(bool(re.search(r'[A-Za-z]{3,}', row['prediction'])) for row in rows),
    }

print('setting               median   p90   capped   empty   rep4>0.1   ASCII word')
for label, rows in [('full greedy', full), ('LoRA sampled', sampled)]:
    s = summarize(rows)
    print(f"{label:22} {s['median']:6} {s['p90']:5} {s['capped']:8} {s['empty']:7} {s['repetitive']:11} {s['ascii']:12}")

setting               median   p90   capped   empty   rep4>0.1   ASCII word
full greedy                96   120        5       0         148            7
LoRA sampled               93   115        7       0          36          241


The aggregate result supports a narrow claim: generation is usually non-empty, moderate in length, and in Nepali script. It does **not** establish factual correctness. The next example shows why.

In [3]:
example = next(row for row in full if row['question'] == 'नेकपा माओवादीका उम्मेदवार कसले दर्ता गराए?')
print('QUESTION\n' + example['question'])
print('\nGOLD\n' + example['gold'])
print('\nFULL FINE-TUNE\n' + example['prediction'])

QUESTION
नेकपा माओवादीका उम्मेदवार कसले दर्ता गराए?

GOLD
नेकपा माओवादीका तर्फबाट अछाम क्षेत्र नम्बर–२ मा शेरबहादुर विकले उम्मेदवारी दर्ता गराएका छन्। उनले मंगलबार दिउँसो ३ बजेसम्म मंगलसेनस्थित मुख्य निर्वाचन अधिकृतको कार्यालयमा पुगेर मनोनयन दर्ता प्रक्रिया पूरा गरेका हुन्। यो निर्वाचनमा सहभागी हुनका लागि गरिने महत्त्वपूर्ण कार्य हो। यसले गर्दा दल र उम्मेदवारहरूको औपचारिक सहभागिता सुनिश्चित हुन्छ।

FULL FINE-TUNE
नेकपा माओवादीका तर्फबाट अछाम क्षेत्र नम्बर–२ मा नारायणप्रसाद भट्टराईले उम्मेदवारी दर्ता गराएका छन्। उनले मंगलबार दिउँसो ३ बजेसम्म मंगलसेनस्थित निर्वाचन अधिकृतको कार्यालयमा पुगेर मनोनयन दर्ता गराएका हुन्। यो निर्वाचनमा सहभागी हुनका लागि दलहरूले उम्मेदवार छनोट गर्ने प्रक्रिया अपनाएका छन्।


Most surrounding details match, but the generated answer replaces **शेरबहादुर विक** with **नारायणप्रसाद भट्टराई**. This is the central limitation: plausible style can conceal a wrong answer.

In [4]:
worst = max(full, key=lambda row: rep4(row['prediction']))
print('QUESTION\n' + worst['question'])
print(f"\nREP-4: {rep4(worst['prediction']):.3f} | generated tokens: {worst['generated_tokens']} | hit cap: {worst['hit_max_new_tokens']}")
print('\nOUTPUT (first 200 characters)\n' + worst['prediction'][:200])

QUESTION
निर्वाचन प्रहरीले के-कस्ता सुविधा पाउँछन्?

REP-4: 0.932 | generated tokens: 256 | hit cap: True

OUTPUT (first 200 characters)
निर्वाचन प्रहरीले सुरक्षाका लागि आवश्यक सबै सुविधा पाउनेछन्। उनीहरूले सुरक्षाकर्मी, प्रहरी, प्रहरी प्रहरी, प्रहरी प्रहरी, प्रहरी प्रहरी, प्रहरी प्रहरी, प्रहरी प्रहरी, प्रहरी प्रहरी, प्रहरी प्रहरी, प्र


## Metric-level comparison

These values come from the full 1,718-turn evaluation artifact. Confidence intervals and entity-category counts remain in `results/generation_metrics.json`.

In [5]:
metrics = json.loads((ROOT / 'results' / 'generation_metrics.json').read_text(encoding='utf-8'))
systems = ['base', 'lora-r16-e3-l1024-seed42', 'full-e3-l1024-seed42', 'lora-seed42-sampled']
print('system                    chrF   ROUGE-L   Nepali   Entity F1   rep-4')
for name in systems:
    m = metrics[name]
    print(f"{name:25} {m['chrf']['mean']:5.2f} {m['rougeL_unicode']['mean']:10.3f} {m['language_ne_accuracy']:8.1%} {m['entity_f1']:11.3f} {m['rep_4']['mean']:8.3f}")

system                    chrF   ROUGE-L   Nepali   Entity F1   rep-4
base                      17.09      0.131    72.4%       0.146    0.065
lora-r16-e3-l1024-seed42  38.07      0.229    99.6%       0.225    0.037
full-e3-l1024-seed42      39.41      0.238   100.0%       0.236    0.029
lora-seed42-sampled       37.20      0.215    99.5%       0.207    0.011


## Conclusion

The full fine-tune improves overlap, language control, and entity recall relative to the base model. Its generated text often reads like Nepali news copy. The evidence does not justify saying that it is factually reliable: entity F1 is only 0.236, the example above changes the answer's person, and 5 greedy outputs hit the token cap with severe repetition. No blinded human evaluation has been run, so fluency is described qualitatively rather than assigned a human score.